In [ ]:
import pandas as pd

y = pd.read_csv('target_vector.csv')
y = y[['Case.ID', 'ProliferationScore']]

X_clinical = pd.read_csv('X_clinical.csv')
X_rppa = pd.read_csv('merged_RPPA.csv')

df_mrna = pd.read_csv('merged_mRNA_TPM.csv')

#rename 
y = y.rename(columns={'Case.ID': 'Case_ID'})
X_rppa = X_rppa.rename(columns={X_rppa.columns[0]:'Case_ID'})
df_mrna = df_mrna.rename(columns={df_mrna.columns[0]:'Case_ID'})

print(y.shape)
print(X_clinical.shape)
print(X_rppa.shape)
print(df_mrna.shape)

Index(['Case_ID', '1433BETA', '1433EPSILON', '1433ZETA', '4EBP1', '4EBP1_pS65',
       '4EBP1_pT37T46', '4EBP1_pT70', '53BP1', 'ACC1',
       ...
       'b-Actin', 'b-Catenin_pT41_S45', 'c-Abl_pY412', 'c-IAP2', 'cGAS',
       'cdc25C', 'eIF4E_pS209', 'p38-a', 'p44-42-MAPK', 'p90RSK_pT573'],
      dtype='str', length=465)
Index(['Case_ID', '5S_rRNA', '5_8S_rRNA', '7SK', 'A1BG', 'A1BG-AS1', 'A1CF',
       'A2M', 'A2M-AS1', 'A2ML1',
       ...
       'ZYG11A', 'ZYG11AP1', 'ZYG11B', 'ZYX', 'ZYXP1', 'ZZEF1', 'ZZZ3',
       'hsa-mir-1253', 'hsa-mir-423', 'snoZ196'],
      dtype='str', length=59428)
Index(['Case_ID', 'ProliferationScore'], dtype='str')
(79, 2)
(79, 9)
(79, 465)
(79, 59428)


In [4]:
y.set_index('Case_ID', inplace=True)
X_clinical.set_index('Case_ID', inplace=True)
X_rppa.set_index('Case_ID', inplace=True)
df_mrna.set_index('Case_ID', inplace=True)

In [5]:
master_IDs = y.index.tolist()

X_clinical = X_clinical.loc[master_IDs]
X_rppa = X_rppa.loc[master_IDs]
df_mrna = df_mrna.loc[master_IDs]

print(f"Target: {len(y)}, mRNA: {len(df_mrna)}, RPPA: {len(X_rppa)}, Clinical: {len(X_clinical)}")

Target: 79, mRNA: 79, RPPA: 79, Clinical: 79


Need to log_2(x+1) transform the mRNA gene expression data

In [8]:
import numpy as np

df_mrna_log = np.log2(df_mrna + 1)
final_mrna = df_mrna_log.loc[:, df_mrna_log.std() > 0.1]

need to handle missing values (imputation) for RPPA data

In [9]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
X_rppa = pd.DataFrame(imputer.fit_transform(X_rppa), 
                             columns=X_rppa.columns, 
                             index=X_rppa.index)

In [10]:
proliferation_leakage_genes = [
    'BIRC5', 'CCNB1', 'CDC20', 'CEP55', 'MKI67', 'NDC80', 
    'NUF2', 'PTTG1', 'RRM2', 'TYMS', 'UBE2C', 
    'CENPF', 'EXO1', 'KIF2C', 'MELK', 'MYBL2', 'ORC6L', 'UBE2T'
]

df_mrna_filtered = final_mrna.drop(columns=proliferation_leakage_genes, errors='ignore')

In [14]:
def parse_gmt_and_map(gmt_filepath, available_genes):
    """
    Parses a .gmt file and maps the genes to the available columns in df_mrna.
    
    Args:
        gmt_filepath (str): Path to the MSigDB .gmt file.
        available_genes (set or list): list(df_mrna.columns)
        
    Returns:
        dict: {pathway_name: [list_of_valid_genes]}
    """
    pathway_dict = {}
    available_set = set(available_genes)
    
    with open(gmt_filepath, 'r') as file:
        for line in file:
            parts = line.strip().split('\t')
            pathway_name = parts[0]
            # parts[1] is the URL/description, parts[2:] are the genes
            genes = set(parts[2:])
            
            # Intersection prevents KeyErrors later when slicing df_mrna
            valid_genes = list(genes.intersection(available_set))
            
            if valid_genes:
                pathway_dict[pathway_name] = valid_genes
                
    return pathway_dict

hallmark_dict = parse_gmt_and_map('hallmark.gmt', list(df_mrna_filtered.columns))
print(len(hallmark_dict))

50


In [15]:
adhesion_genes = ['CDH1', 'CTNNA1', 'CTNNB1', 'CTNND1']
akt_genes = ['PTEN', 'PIK3CA', 'AKT1', 'AKT2', 'AKT3', 'INPP4B', 'EGFR', 'ERBB2', 'STAT3']
tf_genes = ['FOXA1', 'GATA3', 'RUNX1', 'TBX3', 'ESR1']

custom_lists = {
    'ILC_Adhesion': adhesion_genes,
    'ILC_AKT_Pathway': akt_genes,
    'ILC_TF_Drivers': tf_genes
}

pathways_dict = custom_lists | hallmark_dict
print(len(pathways_dict))



53


In [18]:
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel
import numpy as np

def compute_kernels(df, feature_dict, kernel_type='linear', gamma=None):
    """
    Computes an N x N kernel matrix for each feature subset.
    """
    kernels = {}
    for name, genes in feature_dict.items():
        # Ensure we only use genes present in the dataframe
        valid_genes = [g for g in genes if g in df.columns]
        if not valid_genes:
            continue
            
        X_subset = df[valid_genes].values
        
        if kernel_type == 'linear':
            K = linear_kernel(X_subset)
        elif kernel_type == 'rbf':
            # Default gamma is 1 / n_features if not specified
            K = rbf_kernel(X_subset, gamma=gamma)
            
        kernels[name] = K
        
    return kernels

computed_kernels = compute_kernels(df_mrna_filtered, pathways_dict, kernel_type='linear')

In [21]:
def normalize_kernel(K):
    """
    Normalizes a kernel matrix so that all diagonal elements are 1.
    Formula: K_norm(i, j) = K(i, j) / sqrt(K(i, i) * K(j, j))
    """
    # Extract the diagonal elements (self-similarities)
    d = np.diag(K).copy()
    
    # Prevent division by zero for completely flat/empty kernels
    d[d == 0] = 1e-10 
    
    # Compute the inverse square root of the diagonal
    D_inv_sqrt = 1.0 / np.sqrt(d)
    
    # Vectorized normalization: 
    # K_norm = D_inv_sqrt (column vector) * K * D_inv_sqrt (row vector)
    K_norm = K * D_inv_sqrt[:, np.newaxis] * D_inv_sqrt[np.newaxis, :]
    
    return K_norm

def normalize_all_kernels(kernels_dict):
    """
    Iterates through a dictionary of kernels and normalizes each one.
    """
    normalized_kernels = {}
    for name, K in kernels_dict.items():
        normalized_kernels[name] = normalize_kernel(K)
        
    return normalized_kernels

# Apply it to the dictionary you created in the previous step
normalized_computed_kernels = normalize_all_kernels(computed_kernels)

In [22]:
# --- 1. Clinical Kernel ---

# Linear kernel is mathematically ideal for one-hot encoded categorical data
K_clinical = linear_kernel(X_clinical.values)
K_clinical_norm = normalize_kernel(K_clinical)


# --- 2. RPPA Kernel ---

# Compute RBF (Gaussian) Kernel for continuous protein data
# (You can also use linear_kernel if you prefer to assume linear relationships)
K_rppa = rbf_kernel(X_rppa.values)
K_rppa_norm = normalize_kernel(K_rppa)


# --- 3. Merge into your Master Dictionary ---
# Add these to the dictionary of normalized mRNA kernels we made previously
normalized_computed_kernels['Clinical_Global'] = K_clinical_norm
normalized_computed_kernels['RPPA_Global'] = K_rppa_norm

print(f"Successfully added Clinical and RPPA kernels. Total kernels: {len(normalized_computed_kernels)}")

Successfully added Clinical and RPPA kernels. Total kernels: 55


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score

X_combined = pd.concat([X_clinical, X_rppa, df_mrna_filtered], axis=1)

y_target = y['ProliferationScore']

X_train, X_test, y_train, y_test = train_test_split(X_combined, y_target, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Feature matrix shape: {X_combined.shape}")

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Regressor:")
print(f"RMSE:{rmse_rf:.4f}")
print(f"R-squared:{r2_rf:.4f}")

svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr.fit(X_train_scaled, y_train)
y_pred_svr = svr.predict(X_test_scaled)
rmse_svr = root_mean_squared_error(y_test, y_pred_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print("Support Vector Regression (SVR):")
print(f"RMSE:{rmse_svr:.4f}")
print(f"R-squared:{r2_svr:.4f}")

Feature matrix shape: (79, 39158)
Random Forest Regressor:
MSE:0.3277
R-squared:0.4968
Support Vector Regression (SVR):
MSE:0.4115
R-squared:0.2067


In [30]:

from scipy.optimize import minimize
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, root_mean_squared_error

# 1. Prepare the Inputs
# Extract the names and the actual N x N matrices into lists to guarantee consistent ordering
kernel_names = list(normalized_computed_kernels.keys())
K_list = list(normalized_computed_kernels.values())

# y is your PAM50 Proliferation Score array (shape: 79,)
# Ensure y is a 1D numpy array
y_target = y.values.ravel() if hasattr(y, 'values') else np.array(y).ravel()

# 2. Define the Objective Function
def mkl_objective(weights, K_list, y):
    """
    Constructs a combined kernel using the given weights, trains a 
    Kernel Ridge Regression model, and returns the Cross-Validated MSE.
    """
    # Build the combined kernel: sum(w_m * K_m)
    K_combined = np.zeros_like(K_list[0])
    for w, K in zip(weights, K_list):
        K_combined += w * K
        
    # Initialize the base learner
    # Note: We use KernelRidge because it is analytically faster inside an 
    # optimization loop than SVR, but SVR(kernel='precomputed') also works.
    regr = KernelRidge(kernel='precomputed', alpha=1.0)
    
    # Evaluate using 5-Fold Cross Validation
    mse_scorer = make_scorer(root_mean_squared_error)
    scores = cross_val_score(regr, K_combined, y, cv=5, scoring=mse_scorer)
    
    # The optimizer tries to MINIMIZE this return value
    return np.mean(scores)

# 3. Set up the Optimizer Constraints and Bounds
n_kernels = len(K_list)

# Initial guess: equal weighting for all kernels (1 / M)
initial_weights = np.ones(n_kernels) / n_kernels

# Bounds: w_m must be between 0 and 1
bounds = [(0.0, 1.0) for _ in range(n_kernels)]

# Constraints: The sum of all weights must exactly equal 1.0
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})

print("Starting Meta-Learner Optimization. This may take a minute depending on the number of kernels...")

# 4. Run the Optimization
result = minimize(
    mkl_objective, 
    initial_weights, 
    args=(K_list, y_target), 
    method='SLSQP', # Sequential Least SQuares Programming (handles bounds/constraints well)
    bounds=bounds, 
    constraints=constraints,
    options={'disp': True, 'maxiter': 100}
)

# 5. Extract and Interpret the Biological Drivers
optimal_weights = result.x

# Map the optimized weights back to their biological pathway names
pathway_importances = {name: weight for name, weight in zip(kernel_names, optimal_weights)}

# Sort them from highest weight (strongest driver) to lowest
sorted_drivers = dict(sorted(pathway_importances.items(), key=lambda item: item[1], reverse=True))

print("\n--- Top Biological Drivers of ILC Proliferation ---")
for pathway, weight in sorted_drivers.items():
    if weight > 0.01: # Filter out pathways the model completely ignored
        print(f"{pathway}: {weight:.4f}")

Starting Meta-Learner Optimization. This may take a minute depending on the number of kernels...
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.3025427312510863
            Iterations: 18
            Function evaluations: 1008
            Gradient evaluations: 18

--- Top Biological Drivers of ILC Proliferation ---
HALLMARK_SPERMATOGENESIS: 0.5607
HALLMARK_KRAS_SIGNALING_DN: 0.4181
RPPA_Global: 0.0213
